## Create NACC train and test splits

Created by: Sahana Kowshik

Date: 05/06/2025

In [1]:
import pandas as pd
import json

In [20]:
directory_path = "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc"
data = pd.read_csv(f"{directory_path}/csv_to_txt/all_nacc_csv_to_txt.csv")
nacc_np = pd.read_csv(f'/projectnb/vkolagrp/datasets/NACC/csv/processed/investigator_ftldlbd_merged_neuropath_vnum_unique_nacc65.csv')

/scratch/4916585.1.cds/ipykernel_1084740/4100326314.py:2: DtypeWarning: Columns (20,22,24,26,28,40,43,45,47,50,60,62,64,66,68,70,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,155,164,175,178,188,216,219,221,223,225,227,229,231,233,235,237,239,241,243,245,247,249,251,253,255,257,259,261,263,265,267,269,271,396,398,400,418,420,422,431,444,453,492,571,599,607,663,679,693,696,699,716,727,733,800,808,819,821,823,825,831,892,947,948,949,957,958,959,960,970,992,995,998,1017,1022,1192,1196,1199,1395,1397,1399,1400,1402,1409,1411,1413,1414,1421,1423,1425,1427,1428,1435,1450,1464,1478,1492,1494,1530,1546,1548,1550,1552,1554,1556,1558,1560,1562,1564,1566,1568,1570,1572,1574,1576,1578,1580,1582,1584,1586,1588,1590,1592,1594,1596,1598,1600,1650,1651,1653,1654,1657,1658,1661,1662,1665,1666,1669,1670,1744,1803,1812,1814,1816,1818,1829,1831,1833,1841,1843,1845,1847,1855,1857,1859,1861,1887) have mixed types. Specify dtype option on import or set low_memory

In [29]:
len(set(data[data['NACCUDSD'] != 2]['NACCID']))

49938

In [4]:
# data = data[data['NACCUDSD'] != 2].reset_index(drop=True)

In [21]:
train_data = data[~data['NACCID'].isin(nacc_np['NACCID'])].reset_index(drop=True)
np_data = data[data['NACCID'].isin(nacc_np['NACCID'])].reset_index(drop=True)

In [23]:
train_data = train_data[train_data['NACCUDSD'] != 2].reset_index(drop=True)
np_data = np_data[np_data['NACCUDSD'] != 2].reset_index(drop=True)

In [24]:
print(len(train_data))
print(len(np_data))

146943
33442


In [25]:
def get_latest_visits(data):
    result = data.sort_values(by=['NACCID', 'NACCVNUM'], ascending=[True, False])
    result = result.drop_duplicates(subset='NACCID', keep='first').reset_index(drop=True)
    return result

In [26]:
train_latest = get_latest_visits(train_data)
np_latest = get_latest_visits(np_data)

In [27]:
print(len(train_latest))
print(len(np_latest))

42041
7897


In [10]:
# if not os.path.exists(f"{directory_path}/train_data/"):
#     os.makedirs(f"{directory_path}/train_data", exist_ok=True)
    
# train_data.to_csv(f"{directory_path}/train_data/train_all_visits.csv")
# train_latest.to_csv(f"{directory_path}/train_data/train_latest_visits.csv")
# train_1st_last.to_csv(f"{directory_path}/train_data/train_1st_last_visits.csv")

# np_data.to_csv(f"{directory_path}/train_data/np_all_visits.csv")
# np_latest.to_csv(f"{directory_path}/train_data/np_latest_visits.csv")

In [30]:
# train1 = pd.read_csv("/projectnb/vkolagrp/projects/adrd_foundation_model/intermediate_data/nacc_processed_data/train_latest_visits.csv")
# test1 = pd.read_csv('/projectnb/vkolagrp/projects/adrd_foundation_model/intermediate_data/nacc_processed_data/np_latest_visits.csv')

In [31]:
# train1 = train1[train1['NACCUDSD'] != 2].reset_index(drop=True)
# test1 = test1[test1['NACCUDSD'] != 2].reset_index(drop=True)

In [32]:
ids_used = set()

## Major

In [33]:
len(np_latest[(~np_latest['NACCID'].isin(ids_used)) & (np_latest['NACCUDSD'].isin([3,4])) & ((np_latest['NACCLEWY'].isin([0,1,2,3,4])) | (np_latest['NPFTDTAU'].isin([0,1])) | (np_latest['NPADNC'].isin([0,1,2,3])))].sample(frac=1, random_state=0).reset_index(drop=True))

6863

In [34]:
np_df = np_latest[(~np_latest['NACCID'].isin(ids_used)) & (np_latest['NACCUDSD'].isin([3,4])) & ((np_latest['NACCLEWY'].isin([0,1,2,3,4])) | (np_latest['NPFTDTAU'].isin([0,1])) | (np_latest['NPADNC'].isin([0,1,2,3])))].sample(frac=1, random_state=0).reset_index(drop=True).apply(lambda x: x.sample(n=3000, random_state=42))
ids_used = ids_used.union(set(np_df['NACCID']))
len(np_df)

3000

In [38]:
set(np_df['NACCID']).intersection(set(train_latest["NACCID"]))

set()

In [40]:
combined_train = pd.concat([train_latest, np_df], axis=0).sample(frac=1, random_state=0).reset_index(drop=True)
remaining_test = np_latest[~np_latest['NACCID'].isin(combined_train['NACCID'])].sample(frac=1, random_state=0).reset_index(drop=True)

In [45]:
print(f"Length of training data: {len(combined_train)}")
print(f"Length of testing data: {len(remaining_test)}")

Length of training data: 45041
Length of testing data: 4897


In [46]:
print(len(combined_train['NACCID']) + len(remaining_test['NACCID']))

49938


In [44]:
combined_train.to_csv(f"{directory_path}/training_data/train.csv", index=False)
remaining_test.to_csv(f"{directory_path}/training_data/test.csv", index=False)